# 04 · BERT fine-tuning
Fine-tune a pre-trained transformer (default **DistilBERT**) for Positive / Neutral / Negative classification and
compare it with the baselines from notebook 02. Tracked in MLflow; predictions are saved for error analysis (05)
and the dashboard (09).

> ⚠️ **Environment:** this notebook needs `torch`, `transformers`, internet access to download the weights, and
> ideally a GPU (DistilBERT, ~25k reviews, 4 epochs ≈ 10–15 min on a T4; much slower on CPU). If those are missing,
> the notebook **skips the heavy cells** and says so. The training cells below were written against
> `transformers ≥ 4.41` and were **not executed in the authoring sandbox** (no access to Hugging Face) — expect to
> fix small version-specific issues on first run.

**Suggested runs:** (1) text only, raw labels · (2) `USE_RATING=True` · (3) `CLEAN_TRAIN=True` (labels cleaned in 03).
Compare on the **same test set**; report raw-label accuracy as the headline.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, OUT = ROOT / "data" / "processed", ROOT / "outputs"
SEED = 42
LABELS = ["negative", "neutral", "positive"]; L2I = {l: i for i, l in enumerate(LABELS)}
MLFLOW_URI = f"sqlite:///{ROOT / 'mlflow.db'}"

# ---- configuration ---------------------------------------------------------------------------
MODEL_NAME  = "distilbert-base-uncased"     # or "bert-base-uncased", "roberta-base"
MAX_LEN     = 128
EPOCHS      = 4
LR          = 2e-5
BATCH       = 32
USE_RATING  = False      # prepend "Rating: 3.2." to the text (report separately: it is a leakage-like shortcut)
CLEAN_TRAIN = False      # train on data/processed/train_clean.csv (from notebook 03)

try:
    import torch, mlflow
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                              EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
    HAVE_BERT = True
    print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())
except Exception as e:                                # noqa: BLE001
    HAVE_BERT = False
    print("⏭  BERT cells will be skipped — missing dependency:", repr(e)[:120])

def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision_macro": p, "recall_macro": r, "f1_macro": f}

⏭  BERT cells will be skipped — missing dependency: ModuleNotFoundError("No module named 'torch'")


## Data and tokenisation

In [2]:
if HAVE_BERT:
    set_seed(SEED)
    tr = pd.read_csv(PROC / ("train_clean.csv" if CLEAN_TRAIN else "train.csv"))
    va, te = pd.read_csv(PROC / "val.csv"), pd.read_csv(PROC / "test.csv")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    class DS(torch.utils.data.Dataset):
        def __init__(self, df):
            texts = (("Rating: " + df.rating_score.map(lambda x: f"{x:.1f}") + ". " + df.review_text)
                     if USE_RATING else df.review_text).tolist()
            self.enc = tok(texts, truncation=True, max_length=MAX_LEN)
            self.y = df.sentiment_label.map(L2I).tolist()
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            d = {k: v[i] for k, v in self.enc.items()}; d["labels"] = self.y[i]; return d

    ds_tr, ds_va, ds_te = DS(tr), DS(va), DS(te)
    print({"train": len(ds_tr), "val": len(ds_va), "test": len(ds_te)})
    lens = [len(x) for x in ds_tr.enc["input_ids"]]
    print("token length: mean %.0f, 99th pct %.0f (MAX_LEN=%d)" % (np.mean(lens), np.percentile(lens, 99), MAX_LEN))

## Train (early stopping on validation macro-F1)

In [3]:
if HAVE_BERT:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=L2I)
    tag = (f"{MODEL_NAME.split('/')[-1]}_len{MAX_LEN}_lr{LR}_ep{EPOCHS}"
           f"{'_rating' if USE_RATING else ''}{'_clean' if CLEAN_TRAIN else ''}")

    def compute_metrics(ep):
        logits, y = ep
        return {k: float(v) for k, v in metrics(y, logits.argmax(-1)).items()}

    args = TrainingArguments(
        output_dir=str(OUT / "bert_runs" / tag), num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2, weight_decay=0.01,
        warmup_ratio=0.1, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="f1_macro", save_total_limit=1, fp16=torch.cuda.is_available(),
        seed=SEED, report_to="none", logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment("guest-sentiment-bert")
    run = mlflow.start_run(run_name=tag)
    mlflow.log_params(dict(model=MODEL_NAME, max_len=MAX_LEN, epochs=EPOCHS, lr=LR, batch=BATCH,
                           use_rating=USE_RATING, clean_train=CLEAN_TRAIN, n_train=len(tr), seed=SEED))
    t0 = time.time(); trainer.train(); mlflow.log_metric("train_seconds", time.time() - t0)
    hist = pd.DataFrame(trainer.state.log_history)
    for _, h in hist.iterrows():
        for k in ("loss", "eval_loss", "eval_accuracy", "eval_f1_macro"):
            if k in h and pd.notna(h[k]): mlflow.log_metric(k, float(h[k]), step=int(h["step"]))
else:
    print("skipped (no torch/transformers)")

skipped (no torch/transformers)


## Learning curves

In [4]:
if HAVE_BERT:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    t = hist.dropna(subset=["loss"]); e = hist.dropna(subset=["eval_loss"])
    ax[0].plot(t.step, t.loss, label="train loss"); ax[0].plot(e.step, e.eval_loss, "o-", label="val loss"); ax[0].legend(); ax[0].set_title("Loss")
    ax[1].plot(e.step, e.eval_accuracy, "o-", label="val accuracy"); ax[1].plot(e.step, e.eval_f1_macro, "o-", label="val macro-F1")
    ax[1].legend(); ax[1].set_title("Validation metrics"); plt.tight_layout(); plt.show()

## Test-set evaluation and saved predictions

In [5]:
if HAVE_BERT:
    pred = trainer.predict(ds_te)
    probs = torch.softmax(torch.tensor(pred.predictions), -1).numpy(); yhat = probs.argmax(1)
    y_te = te.sentiment_label.map(L2I).values
    mt = metrics(y_te, yhat)
    mlflow.log_metrics({f"test_{k}": float(v) for k, v in mt.items()}); mlflow.end_run()
    print({k: round(v, 4) for k, v in mt.items()})
    out = pd.DataFrame({"review_id": te.review_id, "review_text": te.review_text, "rating_score": te.rating_score,
                        "true": y_te, "pred": yhat, "p_neg": probs[:, 0], "p_neu": probs[:, 1], "p_pos": probs[:, 2]})
    out.to_csv(OUT / f"predictions_bert_{tag}.csv", index=False)
    cm = confusion_matrix(y_te, yhat, labels=[0, 1, 2])
    fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3), LABELS, rotation=30); ax.set_yticks(range(3), LABELS); ax.set(xlabel="Predicted", ylabel="True", title=tag)
    for i in range(3):
        for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout(); plt.savefig(OUT / f"confusion_bert_{tag}.png", dpi=130); plt.show()
    trainer.save_model(str(OUT / "bert_best" / tag)); tok.save_pretrained(str(OUT / "bert_best" / tag))

## Baselines vs BERT (appears once BERT predictions exist)

In [6]:
rows = []
for f in sorted(OUT.glob("predictions_baseline_*.csv")) + sorted(OUT.glob("predictions_bert_*.csv")):
    p = pd.read_csv(f); m = metrics(p.true, p.pred)
    rows.append({"model": f.stem.replace("predictions_", ""), **{k: round(v, 4) for k, v in m.items()}})
pd.DataFrame(rows).sort_values("accuracy", ascending=False) if rows else print("no prediction files yet")

,model,accuracy,precision_macro,recall_macro,f1_macro
1,baseline_text_plus_rating,0.9427,0.9465,0.9421,0.9441
0,baseline_text_only,0.9407,0.9450,0.9399,0.9421


### Interpreting the comparison
On this data every approach is capped by label noise (notebook 03): expect BERT to land close to the best
baseline on raw labels. Its value is (a) a fair comparison for the project report and (b) better handling of negation
and mixed language in the error analysis (05). If BERT is *not* clearly better, that is a legitimate finding:
the simpler model is the better engineering choice.